In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_predict, learning_curve
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score
)
from xgboost import XGBRegressor
def generate_large_dataset():
    RIs         = np.round(np.arange(1.40, 1.471, 0.001), 3)
    wavelengths = np.round(np.arange(0.96, 1.061, 0.002), 3)
    data        = []
    np.random.seed(42)

    for ri in RIs:
        for wl in wavelengths:
            base_re   = 1.4332 - (wl - 0.96) * 0.003
            re_offset = -0.0004 * (ri - 1.40)
            Re_eff    = base_re + re_offset + np.random.normal(0, 0.0001)
            im_base      = -5e-8 * (1 + (ri - 1.40) * 12.0)
            im_wl_factor = 1 + (wl - 0.96) * 2.5
            Im_eff       = im_base * im_wl_factor + np.random.normal(0, 1e-10)
            closs_signal = 8.686 * (2 * np.pi * abs(Im_eff) * 1e6) / wl
            closs        = closs_signal + np.random.normal(0, abs(closs_signal) * 0.01)
            ws = 450 - (ri - 1.40) * 2000 + np.random.normal(0, 5)
            data.append([ri, wl, Re_eff, Im_eff, closs, ws])
    return pd.DataFrame(
        data, columns=["RI", "Wavelength", "Re_eff", "Im_eff", "CLoss", "WS"]
    )
df = generate_large_dataset()
df.to_csv("expanded_dataset.csv", index=False)
print(f"Dataset shape : {df.shape}")
print(f"Total entries : {len(df)}\n")
print(df.head())
df      = pd.read_csv("expanded_dataset.csv")
X       = df.drop(["WS", "CLoss"], axis=1)
y_ws    = df["WS"]
y_closs = df["CLoss"]
scaler   = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
mlr  = LinearRegression()
xgbr = XGBRegressor(n_estimators=100, learning_rate=0.1,
                    max_depth=5, random_state=42,
                    eval_metric='rmse', verbosity=0)
dtr  = DecisionTreeRegressor(max_depth=10, random_state=42)
svr  = SVR(kernel='rbf', C=1e3, gamma=0.1)
rfr  = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
gbr  = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1,
                                  max_depth=5, random_state=42)
hybrid_pair1 = StackingRegressor(
    estimators=[('mlr', mlr), ('xgbr', xgbr)],
    final_estimator=Ridge(alpha=1.0), cv=5
)
hybrid_pair2 = StackingRegressor(
    estimators=[('dtr', dtr), ('svr', svr)],
    final_estimator=Ridge(alpha=1.0), cv=5
)
hybrid_pair3 = StackingRegressor(
    estimators=[('rfr', rfr), ('gbr', gbr)],
    final_estimator=Ridge(alpha=1.0), cv=5
)
models = {
    "MLR + XGBR" : hybrid_pair1,
    "DTR + SVR"  : hybrid_pair2,
    "RFR + GBR"  : hybrid_pair3,
}
closs_predictions = {}
ws_predictions    = {}
for name, model in models.items():
    closs_predictions[name] = cross_val_predict(model, X_scaled, y_closs, cv=10)
    ws_predictions[name]    = cross_val_predict(model, X_scaled, y_ws,    cv=10)
    def compute_full_metrics(y_true, y_pred):
    mape = np.mean(
        np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))
    ) * 100
    return {
        "MAE"      : mean_absolute_error(y_true, y_pred),
        "MSE"      : mean_squared_error(y_true, y_pred),
        "RMSE"     : np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²"       : r2_score(y_true, y_pred),
        "MAPE (%)" : mape,
        "Expl.Var" : explained_variance_score(y_true, y_pred),
    }
def print_metrics(y_true, y_pred, target_name):
    m = compute_full_metrics(y_true, y_pred)
    print(f"\n  {target_name}:")
    print(f"    MAE       : {m['MAE']:.4f}")
    print(f"    MSE       : {m['MSE']:.4f}")
    print(f"    RMSE      : {m['RMSE']:.4f}")
    print(f"    R²        : {m['R²']:.4f}")
    print(f"    MAPE (%)  : {m['MAPE (%)']:.4f}")
    print(f"    Expl.Var  : {m['Expl.Var']:.4f}")
print("=" * 45)
print("  Cross-Validated Performance")
print("=" * 45)
for name in models.keys():
    print(f"\n── {name}")
    print_metrics(y_closs, closs_predictions[name], "CLoss")
    print_metrics(y_ws,    ws_predictions[name],    "WS")
    closs_metrics = {n: compute_full_metrics(y_closs, closs_predictions[n]) for n in models}
ws_metrics    = {n: compute_full_metrics(y_ws,    ws_predictions[n])    for n in models}
df_closs_summary = pd.DataFrame(closs_metrics).T.round(4)
df_ws_summary    = pd.DataFrame(ws_metrics).T.round(4)
print("\n── CLoss Performance Summary ──")
print(df_closs_summary.to_string())
print("\n── WS Performance Summary ──")
print(df_ws_summary.to_string())
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plt.suptitle("Actual vs. Predicted Values", fontsize=16, y=1.02)
for idx, (name, y_pred) in enumerate(closs_predictions.items()):
    ax = axes[0, idx]
    r2 = closs_metrics[name]["R²"]
    ax.scatter(y_closs, y_pred, alpha=0.4, color='steelblue', s=15)
    ax.plot([y_closs.min(), y_closs.max()],
            [y_closs.min(), y_closs.max()], 'r--', lw=1.5)
    ax.set_xlabel("Actual CLoss (dB/cm)", fontsize=11)
    ax.set_ylabel("Predicted CLoss (dB/cm)", fontsize=11)
    ax.set_title(f"{name} — CLoss", fontsize=13)
    ax.annotate(f"R² = {r2:.4f}", xy=(0.05, 0.92),
                xycoords='axes fraction', fontsize=11,
                color='darkred', fontweight='bold')
    ax.grid(True, alpha=0.3)
for idx, (name, y_pred) in enumerate(ws_predictions.items()):
    ax = axes[1, idx]
    r2 = ws_metrics[name]["R²"]
    ax.scatter(y_ws, y_pred, alpha=0.4, color='seagreen', s=15)
    ax.plot([y_ws.min(), y_ws.max()],
            [y_ws.min(), y_ws.max()], 'r--', lw=1.5)
    ax.set_xlabel("Actual WS", fontsize=11)
    ax.set_ylabel("Predicted WS", fontsize=11)
    ax.set_title(f"{name} — WS", fontsize=13)
    ax.annotate(f"R² = {r2:.4f}", xy=(0.05, 0.92),
                xycoords='axes fraction', fontsize=11,
                color='darkred', fontweight='bold')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# ── 9.
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plt.suptitle("Residual Plots (Predicted − Actual)", fontsize=16, y=1.02)

for idx, (name, y_pred) in enumerate(closs_predictions.items()):
    ax        = axes[0, idx]
    residuals = y_pred - y_closs
    ax.scatter(y_pred, residuals, alpha=0.4, color='steelblue', s=15)
    ax.axhline(0, color='red', linestyle='--', lw=1.5)
    ax.set_xlabel("Predicted CLoss (dB/cm)", fontsize=11)
    ax.set_ylabel("Residual", fontsize=11)
    ax.set_title(f"{name} — CLoss Residuals", fontsize=13)
    ax.grid(True, alpha=0.3)

for idx, (name, y_pred) in enumerate(ws_predictions.items()):
    ax        = axes[1, idx]
    residuals = y_pred - y_ws
    ax.scatter(y_pred, residuals, alpha=0.4, color='seagreen', s=15)
    ax.axhline(0, color='red', linestyle='--', lw=1.5)
    ax.set_xlabel("Predicted WS", fontsize=11)
    ax.set_ylabel("Residual", fontsize=11)
    ax.set_title(f"{name} — WS Residuals", fontsize=13)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
def plot_residual_distribution(predictions_dict, y_true, target_name):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plt.suptitle(f"Residual Distribution — {target_name}", fontsize=15)

    for idx, (name, y_pred) in enumerate(predictions_dict.items()):
        residuals = y_pred - y_true.values
        ax        = axes[idx]
        ax.hist(residuals, bins=40, color='steelblue',
                edgecolor='white', alpha=0.8)
        ax.axvline(0, color='red', lw=1.5,
                   linestyle='--', label='Zero')
        ax.axvline(residuals.mean(), color='orange', lw=1.5,
                   linestyle='-',
                   label=f'Mean = {residuals.mean():.3f}')
        ax.set_title(name, fontsize=13)
        ax.set_xlabel("Residual", fontsize=11)
        ax.set_ylabel("Frequency", fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_residual_distribution(closs_predictions, y_closs, "CLoss")
plot_residual_distribution(ws_predictions,    y_ws,    "WS")
def plot_prediction_interval(predictions_dict, y_true, target_name, color):
    fig, axes  = plt.subplots(1, 3, figsize=(18, 5))
    plt.suptitle(f"Prediction Interval — {target_name}", fontsize=15)

    idx_sorted = np.argsort(y_true.values)
    y_sorted   = y_true.values[idx_sorted]

    for i, (name, preds) in enumerate(predictions_dict.items()):
        preds_sorted = preds[idx_sorted]
        std          = (y_sorted - preds_sorted).std()

        ax = axes[i]
        ax.plot(y_sorted,       color='black', lw=1,
                label='Actual',    alpha=0.7)
        ax.plot(preds_sorted,   color=color,   lw=1,
                label='Predicted', alpha=0.8)
        ax.fill_between(range(len(y_sorted)),
                        preds_sorted - 1.96 * std,
                        preds_sorted + 1.96 * std,
                        alpha=0.15, color=color,
                        label='95% interval')
        ax.set_title(name, fontsize=13)
        ax.set_xlabel("Sample Index (sorted)", fontsize=11)
        ax.set_ylabel(target_name, fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_prediction_interval(closs_predictions, y_closs, "CLoss", "steelblue")
plot_prediction_interval(ws_predictions,    y_ws,    "WS",    "seagreen")
metric_keys = ["MAE", "RMSE", "R²"]
model_names = list(models.keys())
x           = np.arange(len(model_names))
colors      = ['steelblue', 'seagreen', 'darkorange']

for target_label, metrics_dict in [("CLoss", closs_metrics),
                                    ("WS",    ws_metrics)]:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    plt.suptitle(f"Model Comparison — {target_label}", fontsize=15)

    for i, metric in enumerate(metric_keys):
        values = [metrics_dict[n][metric] for n in model_names]
        bars   = axes[i].bar(x, values, width=0.5, color=colors)
        axes[i].set_xticks(x)
        axes[i].set_xticklabels(model_names, fontsize=10)
        axes[i].set_title(metric, fontsize=13)
        axes[i].set_ylabel(metric, fontsize=11)
        axes[i].grid(axis='y', alpha=0.3)
        for bar, val in zip(bars, values):
            axes[i].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                f"{val:.4f}", ha='center', va='bottom', fontsize=9
            )

    plt.tight_layout()
    plt.show()
    # ── 13.
def plot_radar(df_metrics, target_name):
    df_norm = df_metrics[["MAE", "RMSE", "R²", "MAPE (%)", "Expl.Var"]].copy()

    for col in ["MAE", "RMSE", "MAPE (%)"]:
        col_min, col_max = df_norm[col].min(), df_norm[col].max()
        df_norm[col]     = 1 - (df_norm[col] - col_min) / (col_max - col_min + 1e-9)

    for col in ["R²", "Expl.Var"]:
        col_min, col_max = df_norm[col].min(), df_norm[col].max()
        df_norm[col]     = (df_norm[col] - col_min) / (col_max - col_min + 1e-9)

    categories = list(df_norm.columns)
    N          = len(categories)
    angles     = [n / float(N) * 2 * np.pi for n in range(N)]
    angles    += angles[:1]
    colors     = ['steelblue', 'seagreen', 'darkorange']

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    for i, (model_name, row) in enumerate(df_norm.iterrows()):
        values = row.tolist() + row.tolist()[:1]
        ax.plot(angles, values, 'o-', lw=2,
                color=colors[i], label=model_name)
        ax.fill(angles, values, alpha=0.08, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=11)
    ax.set_title(f"Radar Chart — {target_name}", fontsize=14, pad=20)
    ax.legend(loc='upper right',
              bbox_to_anchor=(1.3, 1.1), fontsize=10)
    plt.tight_layout()
    plt.show()

plot_radar(df_closs_summary, "CLoss")
plot_radar(df_ws_summary,    "WS")
def plot_learning_curves(models, X, y, target_name):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plt.suptitle(f"Learning Curves — {target_name}", fontsize=15)

    for idx, (name, model) in enumerate(models.items()):
        train_sizes, train_scores, val_scores = learning_curve(
            model, X, y,
            cv=5, n_jobs=-1,
            train_sizes=np.linspace(0.1, 1.0, 8),
            scoring='r2'
        )
        train_mean = train_scores.mean(axis=1)
        val_mean   = val_scores.mean(axis=1)
        train_std  = train_scores.std(axis=1)
        val_std    = val_scores.std(axis=1)

        ax = axes[idx]
        ax.plot(train_sizes, train_mean, 'o-',
                color='steelblue', label='Train R²')
        ax.plot(train_sizes, val_mean,   'o-',
                color='seagreen',  label='Val R²')
        ax.fill_between(train_sizes,
                        train_mean - train_std,
                        train_mean + train_std,
                        alpha=0.1, color='steelblue')
        ax.fill_between(train_sizes,
                        val_mean - val_std,
                        val_mean + val_std,
                        alpha=0.1, color='seagreen')
        ax.set_title(name, fontsize=13)
        ax.set_xlabel("Training Samples", fontsize=11)
        ax.set_ylabel("R²", fontsize=11)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_learning_curves(models, X_scaled, y_closs, "CLoss")
plot_learning_curves(models, X_scaled, y_ws,    "WS")
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(12, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    annot_kws={"size": 10}
)
plt.title("Correlation Heatmap of Sensor Features and Targets", fontsize=16)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


Dataset shape : (3672, 6)
Total entries : 3672

    RI  Wavelength    Re_eff        Im_eff     CLoss          WS
0  1.4       0.960  1.433250 -5.001383e-08  2.861688  457.615149
1  1.4       0.962  1.433171 -5.027341e-08  2.897129  453.837174
2  1.4       0.964  1.433141 -5.044574e-08  2.842693  447.671351
3  1.4       0.966  1.433206 -5.094133e-08  2.828370  447.188562
4  1.4       0.968  1.433075 -5.096858e-08  2.847510  442.938481


# 1. Learning Curve — are your models actually learning or just memorizing?

# 2. Prediction Interval Plot — shows where models are uncertain

# 3. Residual Distribution — reveals if errors are normally distributed

# 4. Comprehensive Metrics Table — MAPE and Explained Variance added

# 5. Radar Chart — best single-figure model comparison